# 립 자연스러움 채점: 비공개 Colab
LaPa 비상업 연구 전용. DB에서 export-colab으로 내보낸 v2 묶음만 사용합니다. 팀원 사진은 업로드하지 않습니다. L4 이상 권장. T4는 4비트와 fp16을 사용하며 bf16을 쓰지 않습니다.

In [ ]:
!nvidia-smi


## 1. 비공개 Drive 마운트 및 데이터 무결성 확인
본인만 접근 가능한 폴더에 묶음을 두고 ZIP SHA-256을 입력하세요. 작업이 끝나면 비공개 데이터와 Colab 임시 파일을 삭제하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, zipfile
DRIVE = Path('/content/drive/MyDrive/lip-rubric-private')
ZIP = DRIVE / 'colab_pack.zip'
EXPECTED_SHA256 = '여기에_담당자가_출력받은_SHA256_입력'
actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual.lower() == EXPECTED_SHA256.lower(), f'SHA-256 불일치: {actual}'
ROOT = Path('/content/lip-rubric-pack'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for name in z.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve()), '잘못된 압축 경로'
    z.extractall(ROOT)
print('검증 완료:', actual)


## 2. 의존성 설치 및 환경 기록
설치에는 네트워크가 필요합니다. 모델은 외부 API가 아닌 Colab GPU에서 직접 실행합니다.

In [ ]:
%pip install 'transformers==4.57.6' 'accelerate==1.12.0' 'bitsandbytes==0.49.0' 'qwen-vl-utils==0.0.14'
import subprocess, sys
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))
with (DRIVE / 'nvidia-smi.txt').open('w') as f: subprocess.run(['nvidia-smi'], stdout=f, check=True)


## 3. 모델 카드 확인 및 버전 기록
judges.json에 2026-11-12에 확인한 리비전이 들어 있습니다. 실행 전에 두 모델 카드를 다시 열어 라이선스가 바뀌지 않았는지 확인하세요. Qwen2.5-VL-7B-Instruct는 Apache-2.0입니다. CLIP은 모델 카드에 라이선스 표기가 없고 배포 서비스 사용이 범위 밖이므로 이 연구 실험에만 씁니다.

In [ ]:
import json
cfg = json.loads((ROOT / 'judges.json').read_text())
for name in ('vlm', 'embedding'):
    print(name, cfg[name]['model_id'], cfg[name]['revision'])
    print('  라이선스:', cfg[name]['license'])
assert all(cfg[k]['revision'] for k in ('vlm', 'embedding')), 'judges.json에 정확한 리비전이 필요합니다'

## 4. 비슷한 3장 평균 시험 (CLIP, VLM 없음)
렌더 사진마다 같은 인물을 제외하고 CLIP으로 가장 비슷한 3장을 찾아, 그 사람 점수 평균이 실제 사람 점수와 맞는지 렌더 전체에서 확인합니다. 로컬 간이 특징(색·밝기·결)으로는 상관 -0.53이었습니다. 몇 분이면 끝납니다.

In [ ]:
knn = DRIVE / 'knn_lofo_clip.json'
subprocess.run([sys.executable, str(ROOT/'judge.py'), '--pack', str(ROOT), '--knn-output', str(knn), '--knn-only'], check=True)
print('저장 위치:', knn)

## 5. 사례 검색 및 채점
입술 확대만 임베딩하고 같은 인물을 제외하여 pool 사례 3개를 고릅니다. A 제로샷, B RAG와 세 가지 사례 순서 반복, C 이웃 평균을 수행합니다. 각 결과는 Drive에 바로 기록되며 같은 설정으로 재실행하면 이어갑니다. L4 기준 1~2시간이 걸릴 수 있습니다.

In [ ]:
result = DRIVE / 'judge_results.csv'
subprocess.run([sys.executable, str(ROOT/'judge.py'), '--pack', str(ROOT), '--output', str(result)], check=True)
print('저장 위치:', result)


## 6. 결과 내려받기
`judge_results.csv`와 `knn_lofo_clip.json`을 담당자의 `research/data/lip_rubric/results/`에 저장하고 `db.py import-judge <CSV경로>`로 넣은 뒤 `analyze.py`를 실행합니다. 원본 이미지와 결과는 비공개로 보관하고 실험 종료 후 정해진 절차에 따라 삭제합니다.